# ISOM 835 · The SHAP Lab: three creative uses of SHAP
**Suffolk University · Sawyer Business School · Fall 2026 · Prof. Hasan Arslan**

**In one sentence:** in Session 9, a SHAP receipt explained **one** prediction. Here the receipts become a tool for **new** questions, each in one line of code.

| | The question | One line | The idea |
|---|---|---|---|
| **1** | "Same risk, but for the same reason?" | `p.reason_groups()` | cluster the bookings by their **receipts**, not their columns (Session 10 meets Session 9) |
| **2** | "Why does group B score higher than group A?" | `p.bridge('hotel', 'Resort Hotel', 'City Hotel')` | split a gap between two groups into clue-by-clue pushes |
| **3** | "Has a case like this happened before?" | `p.precedents(438)` | find the past bookings with the most **similar receipts**, and see what happened to them |

Same hotel, same kit as the Trust Kit notebook. Run top to bottom with Shift + Enter; about 2 minutes.

**One rule for the whole lab:** SHAP explains **the model**, not the world. Every idea here tells you how the model thinks. Only an experiment tells you what happens if you act.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## Step 0 · The kit, the hotel, a model

Exactly as in the Trust Kit: 30,000 bookings from two hotels in Portugal, the three leaking columns dropped, a gradient-boosting model (AUC about 0.945).

In [ ]:
!wget -q -O trust_kit.py https://raw.githubusercontent.com/harslan/isom-835/master/public/session-09/trust_kit.py
from trust_kit import start

hotel = pd.read_csv('https://raw.githubusercontent.com/harslan/isom-835/master/public/data/hotel_bookings.csv').sample(30_000, random_state=835)
p = start(hotel, target='is_canceled', drop=['reservation_status', 'reservation_status_date', 'assigned_room_type'], yes_means='canceled')

---
# Lab 1 · Same risk, different reasons

**The story.** Two patients walk in with the same fever: 39 °C. A good doctor doesn't give them the same medicine. One has the flu, the other an infection. Same number, **different cause**, different treatment.

Our model flags hundreds of bookings as "will cancel". Same verdict. But **why**? If we knew, we could treat each group differently.

**The idea.** In Session 10 we grouped customers with k-means on their RFM numbers. Now we group the risky bookings with k-means on their **SHAP receipts**. Bookings whose receipts look alike are risky **for the same reasons**.

A bonus from Session 10: we don't need to scale anything. Every line of a receipt is already in the **same unit** (points of cancel chance). The receipts already speak one currency.

In [ ]:
p.reason_groups()

**What you should see:** about **700** bookings the model calls "will cancel", in **four** groups with very different reasons:

| group | the main reason | what the hotel could do |
|---|---|---|
| the **agent** group (the biggest) | which travel agent booked it, through an online agency, with no special requests | talk to those agents: payment terms, or a confirmation step |
| the **deposit_type** group | a **non-refundable** deposit (+43 points), and every one of them canceled | these are tour-operator **block** bookings: write group contracts with a release date |
| the **country** group | booked from **Portugal** (+28), far ahead | local guests holding a room "just in case": a reminder a week before arrival |
| the **previous_cancellations** group | the guest **canceled before** (+32) | ask repeat cancellers for a deposit |

The heat map shows the same thing as a picture: each row is a group, each column a clue; red pushes toward "cancel".

> **Remember:** a score tells you **who**. Grouped receipts tell you **why**, and **why** decides the action. *Real name: clustering on SHAP values (sometimes called "supervised clustering", because the groups are shaped by what predicts the outcome).*

---
# Lab 2 · The bridge: explain a gap

**The story.** Your boss looks at last year's numbers: *"City Hotel bookings cancel far more than Resort Hotel bookings. Why?"* An average tells you **how big** the gap is. It doesn't tell you **why**.

Finance teams answer this kind of question with a **bridge**: from last year's profit, +3 for higher prices, −2 for higher costs… to this year's profit. SHAP can build the same bridge for a model. The model's average prediction for each group is *average start + average pushes*, so the gap between two groups splits **exactly** into clue-by-clue differences.

In [ ]:
p.bridge('hotel', 'Resort Hotel', 'City Hotel')

**What you should see:** the model expects about **28%** of Resort bookings and **42%** of City bookings to cancel, which matches what really happened (28% and 41%). The bridge from 28 to 42:
- **deposit type, about +5 points:** the City Hotel takes far more **non-refundable group blocks** (the Lab 1 group);
- **parking, about +3 points:** Resort guests often **drive** and ask for a parking space, and a guest who asks for parking almost never cancels;
- then smaller steps: travel agent, lead time, market segment.

One sentence for the boss is printed at the end.

> ⚠️ **Careful.** "+3 for parking" does not mean *"give City Hotel guests a parking space and cancellations drop"*. It means guests who ask for parking are the committed kind. The bridge explains the model's numbers, not cause and effect.

**Try it:** `p.bridge('arrival_date_month', 'January', 'August')`. The two months end up almost equal, but for **opposite reasons**: August bookings are made further ahead (pushes up) but have fewer non-refundable blocks (pushes down). Same total, different story.

> **Remember:** when someone asks *"why is B higher than A?"*, answer with a bridge, clue by clue.

---
# Lab 3 · Cases like this one

**The story.** Doctors and lawyers think in **precedents**: *"I've seen this before. Here's what happened then."* A manager trusts *"five bookings just like this one all canceled"* far more than *"the model says 90%"*.

**The idea.** Take a booking's receipt and find the **past** bookings (from the training data) whose receipts are **closest** to it: similar **reasons**, not just similar columns. Then show what happened to them.

In [ ]:
p.precedents(438)

**What you should see:** booking 438 (booked about 300 days ahead through an online travel agency, changed twice) and its five closest look-alikes from the past. All five **canceled**. The table shows the three clues that matter most for this booking, so a manager can see *why* they count as look-alikes.

Now a hard case: booking **313**, where the model says about **50%**, a coin flip. This is exactly the kind of booking that lands in the **"not sure"** pile (Session 9).

In [ ]:
p.precedents(313)

**What you should see:** the look-alikes **disagree** (about 3 of 5 canceled). The model's "50%" is honest: cases like this really do go both ways. For a "not sure" booking, these five past cases are what a person should read before deciding.

> **Remember:** next to every prediction, show **cases like this one**. It turns a number into evidence people can check. *Real name: nearest neighbors in SHAP space, a form of case-based (example-based) explanation.*

---
# An honest note: an idea that did not hold up

There is a fourth idea that sounds great: a **drift alarm** that needs no labels. Watch the receipts of new bookings every month; if the **reasons** change, ring the alarm, long before you know who really canceled.

We tested it here. We trained on 2015–2016 and watched each month of 2017, compared with the same month a year earlier. The reasons **did** move most in **January and February** ("canceled before" suddenly mattered about half as much). But the model's accuracy fell in **July and August** (AUC about 0.83, against about 0.90 in spring). On this data, the alarm would have rung at the wrong time.

That doesn't make the idea useless: a change in the reasons is a good prompt to take a look. But it is a **smoke detector, not a crystal ball**. This is also how good analysts work: try the clever idea, check it against what really happened, and report what you found, even when it's "not much".

---
# Your turn

1. **More or fewer groups.** Run `p.reason_groups(k=3)` and `p.reason_groups(k=5)`. Which group merges or splits? Would the hotel act differently on the new group?
2. **Your own bridge.** Pick a column and two of its values, e.g. `p.bridge('customer_type', 'Transient', 'Transient-Party')` or `p.bridge('market_segment', 'Direct', 'Online TA')`. Write the one-sentence answer for the boss.
3. **Your own precedent.** Pick any test booking number (0 to 5,999). Do its look-alikes agree with the model?
4. **A different business.** In the house-price cell below: why are the expensive houses expensive? Are there different *kinds* of expensive?

In [ ]:
# Your turn · 1–3: work here

In [ ]:
# Your turn · 4: the same lab on house prices (the model predicts a number)
houses = pd.read_csv('https://raw.githubusercontent.com/harslan/isom-835/master/public/data/ames_housing.csv')
h = start(houses, target='SalePrice', drop=['Id'])
h.reason_groups(k=3)                     # the top 30% of prices, grouped by why the model prices them high
h.bridge('CentralAir', 'N', 'Y')         # why do houses with central air cost so much more?
h.precedents(0)                          # five past sales that look like house 0, and what they really sold for

## What you learned

1. **Group by reasons, not by columns.** Clustering the receipts splits one big "risky" pile into groups that each need their own action.
2. **Explain a gap with a bridge.** The difference between two groups splits exactly into clue-by-clue pushes.
3. **Show cases like this one.** The nearest past receipts turn a number into evidence a person can check.
4. **Test the clever idea before you trust it.** The drift alarm sounded great, and on this data it rang at the wrong time.

And the rule above all of them: SHAP explains **the model**, not the world.

| what we said | the real name |
|---|---|
| grouping by receipts | clustering on SHAP values ("supervised clustering") |
| the bridge | additive attribution of a difference between groups |
| cases like this one | nearest neighbors in SHAP space (example-based explanation) |
| the drift alarm | explanation drift / feature-attribution monitoring |